In [1]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.optimizers import Adam
import tensorflow.keras
from keras.models import Sequential, Model
from keras.layers import *
from keras.utils import Sequence
from keras.layers import Conv2D, MaxPooling2D
from qkeras import *

from keras.utils import Sequence
from keras.callbacks import CSVLogger
from keras.callbacks import EarlyStopping

import os
import random
from datetime import datetime
import time

pi = 3.14159265359

maxval=1e9
minval=1e-9

2025-10-08 19:52:46.287093: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-08 19:52:46.288767: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-08 19:52:46.309314: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-10-08 19:52:46.309330: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-08 19:52:46.309354: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to regi

In [2]:
from dataloaders.OptimizedDataGenerator_v2p5 import OptimizedDataGenerator
from loss import *
#from models.mlp_encoder_model_nonquantized import *
from models.mlp_encoder_model import *
from AnnealingScheduler import *

In [3]:
seed = 10
tf.random.set_seed(seed)
random.seed(seed)

In [4]:
#dataset_base_dir = "/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/"
dataset_base_dir = "/nas/work/research/smartpix-box/pixelAV_datasets/shuffled/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/"
tfrecords_base_dir = os.path.join(dataset_base_dir, "TFR_files", "2t")

dataset_train_dir = os.path.join(dataset_base_dir, "train_contained")
dataset_validation_dir = os.path.join(dataset_base_dir, "test_contained")
tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train_contained_slim")
tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val_contained_slim")

dirs_to_create = [
    tfrecords_dir_train,
    tfrecords_dir_val,
    dataset_train_dir,
    dataset_validation_dir
]

# Create each directory if it doesn't exist
for directory in dirs_to_create:
    os.makedirs(directory, exist_ok=True)

In [5]:
print(f'Number of training files: {len(os.listdir(dataset_train_dir))}')
print(f'Number of validation files: {len(os.listdir(dataset_validation_dir))}')

Number of training files: 80
Number of validation files: 20


In [6]:
batch_size = 5000
val_batch_size = 5000
train_file_size = len(os.listdir(dataset_train_dir))
val_file_size = len(os.listdir(dataset_validation_dir))

In [7]:
start_time = time.time()
validation_generator = OptimizedDataGenerator(
    dataset_base_dir = dataset_validation_dir,
    file_type = "parquet",
    data_format = "3D",
    batch_size = val_batch_size,
    file_count = val_file_size,
    to_standardize = False, # False when processing manually digitized inputs
    log_compression = False, # False when processing manually digitized inputs
    select_contained = True,
    noise = -1,
    min_threshold = None,
    max_threshold = None,
    include_y_local= False,
    labels_list = ['x-midplane','y-midplane','cotBeta'],
    input_shape = (2,16,16), # (20,16,16),
    transpose = (0,2,3,1),
    shuffle = False, 
    files_from_end = True,

    tfrecords_dir = tfrecords_dir_val,
    use_time_stamps = [0,19],
    max_workers = 2,
    load_from_tfrecords_dir = tfrecords_dir_val
)

print("--- Validation generator %s seconds ---" % (time.time() - start_time))

Loading metadata from /nas/work/research/smartpix-box/pixelAV_datasets/shuffled/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/TFR_files/2t/TFR_val_contained_slim/metadata.json
--- Validation generator 0.07451581954956055 seconds ---


In [8]:
# training generator
start_time = time.time()
training_generator = OptimizedDataGenerator(
    dataset_base_dir = dataset_train_dir,
    file_type = "parquet",
    data_format = "3D",
    batch_size = batch_size,
    file_count = train_file_size,
    to_standardize = False, # False when processing manually digitized inputs
    log_compression = False, # False when processing manually digitized inputs
    select_contained = True,
    noise = -1,
    min_threshold = None,
    max_threshold = None,
    include_y_local= False,
    labels_list = ['x-midplane','y-midplane','cotBeta'],
    input_shape = (2,16,16), # (20,16,16),
    transpose = (0,2,3,1),
    shuffle = False, # True 

    tfrecords_dir = tfrecords_dir_train,
    use_time_stamps = [0,19],
    max_workers = 2,
    load_from_tfrecords_dir = tfrecords_dir_train
)
print("--- Training generator %s seconds ---" % (time.time() - start_time))

Loading metadata from /nas/work/research/smartpix-box/pixelAV_datasets/shuffled/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/TFR_files/2t/TFR_train_contained_slim/metadata.json
--- Training generator 0.07739067077636719 seconds ---


In [9]:
training_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir = tfrecords_dir_train,
    shuffle = True,
    seed = seed,
    quantize = False
)

validation_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir = tfrecords_dir_val,
    shuffle = True,
    seed = seed,
    quantize = False
)


Loading metadata from /nas/work/research/smartpix-box/pixelAV_datasets/shuffled/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/TFR_files/2t/TFR_train_contained_slim/metadata.json


Loading metadata from /nas/work/research/smartpix-box/pixelAV_datasets/shuffled/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/TFR_files/2t/TFR_val_contained_slim/metadata.json


In [10]:
model=CreateModel_Slim_SoftQuantizer((16,16,2), initial_thresholds=[400, 1000, 2000], threshold_offset=0.0)
model.compile(
    optimizer=tf.keras.optimizers.Nadam(learning_rate=1e-3),
    loss=custom_sse_loss
)

model.summary()

2025-10-08 19:52:48.084032: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:268] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


Model: "smrtpxl_regression"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_pxls (InputLayer)     [(None, 16, 16, 2)]          0         []                            
                                                                                                  
 soft_quantizer_output (Sof  (None, 16, 16, 2)            8         ['input_pxls[0][0]']          
 tQuantizeLayer)                                                                                  
                                                                                                  
 average_pooling2d (Average  (None, 16, 1, 2)             0         ['soft_quantizer_output[0][0]'
 Pooling2D)                                                         ]                             
                                                                                 

In [11]:
# training
pitch = '50x12P5'
fingerprint = '%08x' % random.randrange(16**8)
#base_dir = '/data/dajiang/smart-pixels/weights/dataset_3src_16x16_weights/'
base_dir = 'weights/dataset_3src_16x16_weights/'
weights_dir = base_dir + 'weights-{}-bs{}-{}-2t-mlp_SLIM-soft_quantizer-checkpoints'.format(pitch, batch_size, fingerprint)

# create output directories
if os.path.isdir(base_dir):
    os.mkdir(weights_dir)
else:
    os.mkdir(base_dir)
    os.mkdir(weights_dir)
    
checkpoint_filepath = weights_dir + '/weights.{epoch:02d}-t{loss:.2f}-v{val_loss:.2f}.hdf5'
mcp = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_loss',
    save_best_only=False,
)

print('Model fingerprint: {}'.format(fingerprint))

Model fingerprint: 34c2da80


In [12]:
scheduler_callback = AnnealingScheduler(
    schedule='cosine',  
    target_layer_name='soft_quantizer_output', 
    initial_k=1.0,
    final_k=67.0, 
    verbose=1      
)

In [13]:
history = model.fit(x=training_generator,
                    validation_data=validation_generator,
                    callbacks=[mcp, scheduler_callback],
                    epochs=2000,
                    shuffle=False, # shuffling now occurs within the data-loader
                    verbose=1)


Epoch 1: Annealing 'k' set to 1.0000
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
Epoch 1/2000
84/84 [==============================] - 8s 76ms/step - loss: 892.0786 - val_loss: 431.6721

Epoch 2: Annealing 'k' set to 1.0000
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
Epoch 2/2000
84/84 [==============================] - 6s 70ms/step - loss: 284.8392 - val_loss: 196.9709

Epoch 3: Annealing 'k' set to 1.0002
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
Epoch 3/2000
84/84 [==============================] - 6s 71ms/step - loss: 171.0921 - val_loss: 147.9673

Epoch 4: Annealing 'k' set to 1.0004
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
Epoch 4/2000
84/84 [==============================] - 6s 72ms/step - loss: 135.3081 - val_loss: 123.2961

Epoch 5: Annealing 'k' set to 1.0007
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
Epoch 5/2000
84/84 [==============================] - 6s 72ms/step - loss: 116.0754 - val_loss: 108.0001

Epoch 6: Annealing 'k' set to 1.0010
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
Epoch 6